# 03 - Feature Engineering

Turn calibrated VV/VH pixels into a compact per-pixel feature vector: raw backscatter, polarimetric ratio/difference (surface scattering type), and local texture statistics (structural context). This feature vector is what both the classical model (04) and the quantum kernel (05) consume - keeping it identical across both is what makes the classical-vs-quantum comparison in notebook 06 fair.

In [1]:
import sys
sys.path.insert(0, r"d:\project-raw-data\sphoorthq-geoverse")

from src.ai.classic.sen1floods11_dataset import load_split, chip_id_from_s1_filename, read_s1, read_label
from src.fusion.pixel_features import FEATURE_NAMES, build_feature_cube, cube_to_pixel_table
from src.observability.run_logger import RunLogger

logger = RunLogger("03_feature_engineering")

In [2]:
with logger.stage("build_features_one_chip") as stage:
    train_pairs = load_split("train")
    s1_filename, _ = train_pairs[0]
    chip_id = chip_id_from_s1_filename(s1_filename)
    s1 = read_s1(chip_id)
    label = read_label(chip_id)
    feature_cube = build_feature_cube(s1)
    stage.metrics = {"chip_id": chip_id, "feature_channels": len(FEATURE_NAMES)}

print(f"feature_cube shape={feature_cube.shape} (channels={FEATURE_NAMES})")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {name:15s} range=[{feature_cube[i].min():.3f}, {feature_cube[i].max():.3f}]")

[03_feature_engineering] -> build_features_one_chip ...
[03_feature_engineering] <- build_features_one_chip [OK] 0.053s {'chip_id': 'Ghana_103272', 'feature_channels': 6}
feature_cube shape=(6, 512, 512) (channels=['vv', 'vh', 'ratio', 'difference', 'vv_local_mean', 'vv_local_std'])
  vv              range=[0.210, 1.000]
  vh              range=[0.051, 0.953]
  ratio           range=[0.000, 0.335]
  difference      range=[0.000, 1.000]
  vv_local_mean   range=[0.525, 0.922]
  vv_local_std    range=[0.130, 1.000]


## Pixel-level feature table (valid pixels only)

Flattens the feature cube to one row per valid pixel (excludes the `-1` no-data label), used directly by notebook 04 (classical) and, subsampled, by notebook 05 (quantum - kernel methods are O(n^2), so QML only ever sees a small subsample).

In [ ]:
with logger.stage("flatten_to_pixel_table") as stage:
    x, y = cube_to_pixel_table(feature_cube, label, raw_s1=s1)
    stage.metrics = {"n_valid_pixels": int(len(y)), "water_fraction": round(float(y.mean()), 3)}

print(f"{len(y)} valid pixels, {y.mean()*100:.1f}% water")

In [4]:
logger.finalize()

[03_feature_engineering] run complete in 0.081s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\8b310506-8371-4ffc-b303-dd20fa41d35d.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\8b310506-8371-4ffc-b303-dd20fa41d35d.json'